In [7]:
"""
CFL-Phish: Dataset Label Audit Script
======================================

This script inspects the label distribution of the Phish360 dataset
to identify potential data quality issues before feature extraction.

Purpose:
    - Verify label consistency across trainval and test splits
    - Identify corrupted or malformed label files
    - Document the distribution of phishing brand targets
    - Provide evidence of data quality assessment for reproducibility

Expected Output:
    - Label frequency distribution for each split
    - Summary statistics (total samples, unique labels, error rates)
    - Examples of each unique label for manual inspection

Usage:
    python inspect_labels.py

    Or adjust paths at the top of the script if running outside Colab.

Note:
    This script is READ-ONLY and does not modify any files.
"""

import os
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple


# ==============================================================================
# Configuration
# ==============================================================================

# Default paths (adjust for your environment)
DEFAULT_TRAINVAL_PATH = "/content/Phish360/trainval"
DEFAULT_TEST_PATH = "/content/Phish360/test"


# ==============================================================================
# Label Inspection Function
# ==============================================================================

def inspect_labels(folder_path: str) -> Tuple[Counter, Dict[str, Tuple[str, str]]]:
    """
    Inspect label distribution in a Phish360 dataset folder.

    Args:
        folder_path: Path to the dataset split folder (trainval or test)

    Returns:
        counter: Frequency count of each unique label content
        examples: Dictionary mapping each label to (folder_name, label_file_path)
    """
    counter = Counter()
    examples = {}

    if not os.path.exists(folder_path):
        print(f"️  Path does not exist: {folder_path}")
        return counter, examples

    folders = sorted([
        folder
        for folder in os.listdir(folder_path)
        if os.path.isdir(os.path.join(folder_path, folder))
    ])

    for folder in folders:
        base_path = os.path.join(folder_path, folder)
        label_path = os.path.join(base_path, "Label")

        # Check for missing Label directory
        if not os.path.exists(label_path):
            counter["NO_LABEL_DIRECTORY"] += 1
            continue

        files = sorted(os.listdir(label_path))

        # Check for empty Label directory
        if not files:
            counter["EMPTY_LABEL_DIRECTORY"] += 1
            continue

        label_file = os.path.join(label_path, files[0])

        # Read label content
        try:
            with open(label_file, "r", errors="ignore") as f:
                content = f.read().strip().lower()
        except Exception:
            counter["READ_ERROR"] += 1
            continue

        # Normalize whitespace
        content = " ".join(content.split())

        # Handle empty content (BOM-only files)
        if not content or content == "\ufeff":
            content = "<EMPTY_OR_BOM_ONLY>"

        counter[content] += 1

        # Store first example of each label
        if content not in examples:
            examples[content] = (folder, label_file)

    return counter, examples


# ==============================================================================
# Summary Statistics
# ==============================================================================

def print_summary(counter: Counter, split_name: str):
    """Print summary statistics for a dataset split."""
    total = sum(counter.values())
    unique_labels = len(counter)
    error_count = (
        counter.get("NO_LABEL_DIRECTORY", 0) +
        counter.get("EMPTY_LABEL_DIRECTORY", 0) +
        counter.get("READ_ERROR", 0) +
        counter.get("<EMPTY_OR_BOM_ONLY>", 0)
    )

    print(f"\n{'=' * 80}")
    print(f"SUMMARY: {split_name}")
    print(f"{'=' * 80}")
    print(f"Total samples          : {total}")
    print(f"Unique labels          : {unique_labels}")
    print(f"Error/corrupted files  : {error_count} ({100*error_count/max(total,1):.2f}%)")
    print(f"Valid brand labels     : {total - error_count} ({100*(total-error_count)/max(total,1):.2f}%)")
    print(f"{'=' * 80}")


# ==============================================================================
# Main Execution
# ==============================================================================

def main():
    """Run label inspection on both trainval and test splits."""

    # Use environment variables or defaults
    trainval_path = os.environ.get("TRAINVAL_PATH", DEFAULT_TRAINVAL_PATH)
    test_path = os.environ.get("TEST_PATH", DEFAULT_TEST_PATH)

    print("=" * 80)
    print("CFL-Phish: Dataset Label Audit")
    print("=" * 80)
    print(f"Trainval path: {trainval_path}")
    print(f"Test path    : {test_path}")

    # Inspect TrainVal
    train_counter, train_examples = inspect_labels(trainval_path)

    print("\n" + "=" * 80)
    print("TRAIN/VAL LABEL DISTRIBUTION (Top 50)")
    print("=" * 80)
    for label, count in train_counter.most_common(50):
        example_info = train_examples.get(label, ("", ""))
        print(f"{repr(label):40s} : {count:5d}  |  Example: {example_info[0]}")

    print_summary(train_counter, "TRAIN/VAL")

    # Inspect Test
    test_counter, test_examples = inspect_labels(test_path)

    print("\n" + "=" * 80)
    print("TEST LABEL DISTRIBUTION (Top 50)")
    print("=" * 80)
    for label, count in test_counter.most_common(50):
        example_info = test_examples.get(label, ("", ""))
        print(f"{repr(label):40s} : {count:5d}  |  Example: {example_info[0]}")

    print_summary(test_counter, "TEST")

    # Combined Statistics
    combined = train_counter + test_counter

    print("\n" + "=" * 80)
    print("COMBINED LABEL DISTRIBUTION (Top 50)")
    print("=" * 80)
    for label, count in combined.most_common(50):
        print(f"{repr(label):40s} : {count:5d}")

    print_summary(combined, "COMBINED")

    # Final Notes
    print("\n" + "=" * 80)
    print("NOTES FOR REVIEWERS")
    print("=" * 80)
    print("""
1. Labels containing brand names (e.g., 'facebook', 'paypal') indicate
   phishing samples targeting those brands.

2. Labels containing 'legitimate' indicate legitimate (non-phishing) samples.

3. The '<EMPTY_OR_BOM_ONLY>' category represents files that contain only
   a Byte Order Mark (BOM) character or are completely empty. These are
   treated as corrupted and excluded during feature extraction.

4. Rare brand labels (appearing < 5 times) are valid phishing samples
   from less common target brands in the Phish360 dataset.

5. This audit confirms data integrity before proceeding with feature
   extraction and model training.
""")


if __name__ == "__main__":
    main()

CFL-Phish: Dataset Label Audit
Trainval path: /content/Phish360/trainval
Test path    : /content/Phish360/test

TRAIN/VAL LABEL DISTRIBUTION (Top 50)
'\ufefflegitimate'                       :  4911  |  Example: L01562_legitimate
'\ufefffacebook'                         :   215  |  Example: P10032_facebook
'\ufeffpaypal'                           :   209  |  Example: P10090_paypal
'\ufeffchase'                            :   179  |  Example: P10022_chase
'\ufeffoffice'                           :   178  |  Example: P10024_office
'\ufeffboa'                              :   130  |  Example: P10015_boa
'\ufefforange'                           :   128  |  Example: P10005_orange
'\ufeffwellsfargo'                       :   117  |  Example: P10185_wellsfargo
'\ufeffdhl'                              :   113  |  Example: P10086_dhl
'\ufeffapple'                            :   102  |  Example: P10099_apple
'\ufeffalibaba'                          :    99  |  Example: P10030_alibaba
'\ufeffitau